<a href="https://colab.research.google.com/github/Timmythaw/langgraph-adk-edu-comparison/blob/main/notebooks/03_llm_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Timmythaw/langgraph-adk-edu-comparison/blob/main/notebooks/03_llm_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-as-a-Judge Evaluation

Uses **Gemini 2.5 Pro** as the judge to evaluate and compare **LangGraph** vs **Google ADK** teacher-assistant outputs across 3 educational scenarios.

| Scenario | Task |
|---|---|
| S1 | Lesson Plan Generation |
| S2 | Quiz Generation |
| S3 | Email Drafting with HITL |

**Rubric:** 5 dimensions (1–5 scale) — Accuracy, Completeness, Clarity, Relevance, Edu Quality.
All judge calls are traced to LangSmith under `langgraph-adk-edu-comparison`.

## 1. Setup

#### Install Dependencies

In [1]:
!pip install langchain-google-genai langsmith pandas matplotlib -q
!pip show langchain-google-genai | grep Version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.3 MB/s eta 0:00:00
Version: 4.2.2


#### Load Secrets

In [ ]:
from google.colab import userdata
import os

PROJECT_ID        = userdata.get('GOOGLE_CLOUD_PROJECT')
LOCATION          = userdata.get('GOOGLE_CLOUD_LOCATION')
LANGSMITH_API_KEY = userdata.get('LANGSMITH_API_KEY')

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY']    = LANGSMITH_API_KEY
os.environ['LANGCHAIN_PROJECT']    = 'langgraph-adk-edu-comparison'
os.environ['LANGCHAIN_ENDPOINT']   = 'https://api.smith.langchain.com'

print('Project:', PROJECT_ID[:15])
print('LangSmith tracing: enabled')

#### Google Authentication

In [ ]:
from google.colab import auth
auth.authenticate_user()

import vertexai
vertexai.init(project=PROJECT_ID, location='us-central1')
print('Authenticated & Vertex AI initialised')

## 2. Judge LLM Initialisation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Use gemini-2.5-pro with extended thinking for fair, reasoned evaluation
judge_llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-pro',
    google_api_key=None,
    vertexai=True,
    project=PROJECT_ID,
    location='us-central1',
    max_output_tokens=2048,
    model_kwargs={'thinking': {'thinking_budget': 2048}},
)
print('Judge: gemini-2.5-pro (thinking_budget=2048)')

## 3. Rubric & Scenarios

#### Scoring Rubric

| # | Dimension | What is evaluated |
|---|---|---|
| 1 | **accuracy** | Factually correct, no hallucinations |
| 2 | **completeness** | All required components present |
| 3 | **clarity** | Well-structured, easy to read |
| 4 | **relevance** | Directly answers the instructor's request |
| 5 | **edu_quality** | Useful for real university teaching at MFU |

In [ ]:
RUBRIC = """
You are an expert evaluator for AI teacher-assistant systems at Mae Fah Luang University.

Score the AI-generated response on 5 dimensions (1-5 each):

1. accuracy       - Factually correct, no hallucinations, grounded in the request
2. completeness   - All required components are present
3. clarity        - Well-structured, logical flow, easy to read
4. relevance      - Directly addresses the instructor's specific request
5. edu_quality    - Genuinely useful for university teaching; pedagogically sound

Scoring scale:
  5 = Excellent | 4 = Good | 3 = Acceptable | 2 = Poor | 1 = Very Poor

Return ONLY valid JSON (no markdown fences):
{{
  "reasoning": "<2-3 sentence chain-of-thought>",
  "scores": {{
    "accuracy": <1-5>,
    "completeness": <1-5>,
    "clarity": <1-5>,
    "relevance": <1-5>,
    "edu_quality": <1-5>
  }},
  "composite": <float, mean of 5 scores>,
  "strengths": "<one sentence>",
  "weaknesses": "<one sentence or null>"
}}
"""
print('Rubric defined.')

#### Test Scenarios

Same 3 prompts used in notebooks `01_langgraph_system` and `02_adk_system`.

In [ ]:
SCENARIOS = {
    'S1 - Lesson Plan': {
        'prompt': (
            'Create a 90-minute lesson plan for first week on Software Testing '
            'for second-year Software Engineering students at Mae Fah Luang University. '
            'Align it with the course materials.'
        ),
        'expected_components': [
            'Learning Objectives', 'Duration/timing breakdown',
            'Teaching methods', 'Assessment strategy', 'Required materials'
        ]
    },
    'S2 - Quiz Generation': {
        'prompt': (
            'Generate 5 multiple-choice questions on Software Testing '
            'from the course materials.'
        ),
        'expected_components': [
            '5 questions', '4 options each (A/B/C/D)',
            'Correct answer marked', 'Explanations'
        ]
    },
    'S3 - Email HITL': {
        'prompt': (
            'Draft and send an email to all students reminding them the '
            'Software Testing quiz covering Unit Testing and Black Box Testing '
            'is next Monday at 9am. Include topics to study and how to prepare.'
        ),
        'expected_components': [
            'Subject line', 'Professional salutation',
            'Quiz date/time', 'Topics to study', 'Preparation guidance'
        ]
    }
}

print(f'Scenarios: {list(SCENARIOS.keys())}')

## 4. Load Experiment Responses

Fetch the actual outputs logged during LangGraph and ADK experiments from LangSmith.

In [ ]:
from langsmith import Client
import pandas as pd

ls = Client()

def fetch_runs(scenario_name: str, framework: str, limit: int = 5) -> list:
    """Fetch runs from LangSmith for a given scenario + framework."""
    runs = list(ls.list_runs(
        project_name='langgraph-adk-edu-comparison',
        run_type='chain',
        filter=(
            f'and(eq(metadata_key, "scenario"), eq(metadata_value, "{scenario_name}"), '
            f'eq(metadata_key, "framework"), eq(metadata_value, "{framework}"))'
        ),
        limit=limit,
    ))
    result = []
    for r in runs:
        latency = None
        if r.end_time and r.start_time:
            latency = round((r.end_time - r.start_time).total_seconds(), 2)
        output_text = ''
        if r.outputs:
            output_text = r.outputs.get('final_output', str(r.outputs))[:3000]
        result.append({
            'run_id': str(r.id), 'framework': framework,
            'scenario': scenario_name, 'output': output_text, 'latency': latency,
        })
    print(f'  {len(result)} runs -- {scenario_name} | {framework}')
    return result

all_runs = []
for sc in SCENARIOS:
    for fw in ['LangGraph', 'ADK']:
        all_runs.extend(fetch_runs(sc, fw))

print(f'\nTotal runs: {len(all_runs)}')

#### CSV Fallback

> Run this cell **only if** `all_runs` is empty above (LangSmith returned 0 results).
> Upload `langgraph_metrics_v2.csv` and `adk_metrics_v2.csv` from the experiment notebooks first.

In [ ]:
# --- CSV FALLBACK: run only if all_runs is empty ---
FALLBACK_FILES = {
    'LangGraph': 'langgraph_metrics_v2.csv',
    'ADK':       'adk_metrics_v2.csv',
}

if not all_runs:
    print('Attempting CSV fallback...')
    for fw, fname in FALLBACK_FILES.items():
        if os.path.exists(fname):
            df_csv = pd.read_csv(fname)
            for i, row in df_csv.iterrows():
                all_runs.append({
                    'run_id': f'{fw}-{i}', 'framework': fw,
                    'scenario': row['scenario'],
                    'output': str(row.get('response', ''))[:3000],
                    'latency': row.get('latency_sec'),
                })
            print(f'  Loaded {len(df_csv)} rows from {fname}')
        else:
            print(f'  {fname} not found')
    print(f'Total after fallback: {len(all_runs)}')
else:
    print('Using LangSmith data -- CSV fallback skipped.')

## 5. Judge Evaluation Function

In [ ]:
import json, re, time
from langsmith import traceable
from langchain_core.messages import HumanMessage

@traceable(
    name='llm_judge_score',
    run_type='chain',
    metadata={'framework': 'Judge', 'model': 'gemini-2.5-pro'}
)
def judge_response(
    scenario_name: str,
    framework: str,
    instructor_prompt: str,
    system_response: str,
    expected_components: list,
) -> dict:
    """Score one system response using Gemini 2.5 Pro as judge."""
    components_str = '\n'.join(f'  - {c}' for c in expected_components)
    prompt = (
        f'{RUBRIC}\n'
        f'=== EVALUATION REQUEST ===\n'
        f'Framework:  {framework}\n'
        f'Scenario:   {scenario_name}\n\n'
        f'Instructor Request:\n{instructor_prompt}\n\n'
        f'Expected Components:\n{components_str}\n\n'
        f'System Response (truncated to 3000 chars):\n{system_response[:3000]}\n\n'
        'Return your evaluation JSON now:'
    )
    raw = judge_llm.invoke([HumanMessage(content=prompt)]).content.strip()
    # Strip markdown code fences if present
    raw = re.sub(r'^```(?:json)?\s*', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\s*```$',          '', raw, flags=re.MULTILINE)
    try:
        result = json.loads(raw)
        result.update({'framework': framework, 'scenario': scenario_name})
        return result
    except json.JSONDecodeError as e:
        print(f'  [JSON parse error] {e}')
        return {'framework': framework, 'scenario': scenario_name, 'error': str(e), 'raw': raw[:300]}

print('Judge function defined.')

#### Run Judge on All Responses

> ~30 judge calls total (5 runs x 3 scenarios x 2 frameworks).
> 5-second sleep between calls to avoid Gemini rate limits.

In [ ]:
judge_results = []

print('Starting LLM-as-a-Judge evaluation...')
print('-' * 60)

for i, run in enumerate(all_runs):
    sc_name  = run['scenario']
    fw       = run['framework']
    response = run['output']

    if not response or len(response.strip()) < 10:
        print(f'  [{i+1}/{len(all_runs)}] SKIP -- empty response: {sc_name} | {fw}')
        continue

    sc = SCENARIOS.get(sc_name)
    if not sc:
        print(f'  [{i+1}/{len(all_runs)}] SKIP -- unknown scenario: {sc_name}')
        continue

    print(f'  [{i+1}/{len(all_runs)}] Judging: {sc_name} | {fw}')
    try:
        result = judge_response(
            scenario_name=sc_name,
            framework=fw,
            instructor_prompt=sc['prompt'],
            system_response=response,
            expected_components=sc['expected_components'],
        )
        result['run_id']  = run.get('run_id')
        result['latency'] = run.get('latency')
        judge_results.append(result)
        s = result.get('scores', {})
        print(f'    composite={result.get("composite", "N/A")} | {s}')
    except Exception as e:
        print(f'    ERROR: {e}')
        judge_results.append({'framework': fw, 'scenario': sc_name, 'error': str(e)})

    if i < len(all_runs) - 1:
        time.sleep(5)  # avoid rate limits

print(f'\nDone. {len(judge_results)} results collected.')

## 6. Aggregate Results

In [ ]:
import pandas as pd

DIMS = ['accuracy', 'completeness', 'clarity', 'relevance', 'edu_quality']

rows = []
for r in judge_results:
    if 'error' in r and 'scores' not in r:
        continue
    s = r.get('scores', {})
    rows.append({
        'framework':    r.get('framework'),
        'scenario':     r.get('scenario'),
        'accuracy':     s.get('accuracy'),
        'completeness': s.get('completeness'),
        'clarity':      s.get('clarity'),
        'relevance':    s.get('relevance'),
        'edu_quality':  s.get('edu_quality'),
        'composite':    r.get('composite'),
        'latency':      r.get('latency'),
        'strengths':    r.get('strengths'),
        'weaknesses':   r.get('weaknesses'),
        'reasoning':    r.get('reasoning'),
    })

df = pd.DataFrame(rows)

print('=' * 65)
print('  Per-Scenario Summary (mean scores)')
print('=' * 65)
summary = df.groupby(['framework', 'scenario'])[DIMS + ['composite']].mean().round(2)
print(summary.to_string())

print('\n' + '=' * 65)
print('  Overall Framework Summary')
print('=' * 65)
overall = df.groupby('framework')[DIMS + ['composite']].mean().round(2)
print(overall.to_string())

In [ ]:
df.to_csv('judge_results.csv', index=False)
print('Saved: judge_results.csv')

from google.colab import files
files.download('judge_results.csv')

## 7. Visualisations

#### Bar Chart — Scores per Dimension per Scenario

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

SCENARIOS_LIST = ['S1 - Lesson Plan', 'S2 - Quiz Generation', 'S3 - Email HITL']
FRAMEWORKS     = ['LangGraph', 'ADK']
COLORS         = {'LangGraph': '#4C72B0', 'ADK': '#55A868'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, scenario in zip(axes, SCENARIOS_LIST):
    sub = df[df['scenario'] == scenario]
    x, width = np.arange(len(DIMS)), 0.35
    for j, fw in enumerate(FRAMEWORKS):
        fw_sub = sub[sub['framework'] == fw]
        means  = [fw_sub[d].mean() if not fw_sub.empty else 0 for d in DIMS]
        bars   = ax.bar(x + j*width - width/2, means, width,
                        label=fw, color=COLORS[fw], alpha=0.85, zorder=3)
        for bar, val in zip(bars, means):
            if val:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                        f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_title(scenario, fontsize=10, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([d.replace('_', '\n') for d in DIMS], fontsize=8)
    ax.set_ylim(0, 5.8)
    ax.set_ylabel('Score (1-5)', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
    ax.set_facecolor('#f9f9f9')

handles = [mpatches.Patch(color=COLORS[fw], label=fw) for fw in FRAMEWORKS]
fig.legend(handles=handles, loc='upper center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('LLM-as-a-Judge: LangGraph vs ADK -- Quality Scores', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('judge_scores_chart.png', dpi=150, bbox_inches='tight')
plt.show()
files.download('judge_scores_chart.png')

#### Radar Chart — Overall Quality Profile

In [ ]:
DIMS_LABELS = ['Accuracy', 'Completeness', 'Clarity', 'Relevance', 'Edu Quality']
N      = len(DIMS_LABELS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for fw, color in COLORS.items():
    fw_df = df[df['framework'] == fw]
    if fw_df.empty: continue
    means = [fw_df[d].mean() for d in DIMS] + [fw_df[DIMS[0]].mean()]
    ax.plot(angles, means, color=color, linewidth=2, label=fw)
    ax.fill(angles, means, color=color, alpha=0.15)

ax.set_thetagrids(np.degrees(angles[:-1]), DIMS_LABELS, fontsize=10)
ax.set_ylim(0, 5)
ax.set_yticks([1,2,3,4,5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=7, color='grey')
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_title('Overall Quality Profile\nLangGraph vs ADK', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('judge_radar.png', dpi=150, bbox_inches='tight')
plt.show()
files.download('judge_radar.png')

#### Scatter — Quality vs Latency

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for fw, color in COLORS.items():
    sub = df[(df['framework'] == fw) & df['latency'].notna() & df['composite'].notna()]
    ax.scatter(sub['latency'], sub['composite'], c=color, label=fw, s=80, alpha=0.8, zorder=3)
ax.set_xlabel('Latency (seconds)', fontsize=11)
ax.set_ylabel('Composite Score (1-5)', fontsize=11)
ax.set_title('Quality vs Speed -- LangGraph vs ADK\n(each point = one run)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('judge_scatter.png', dpi=150)
plt.show()
files.download('judge_scatter.png')

## 8. Head-to-Head Verdict

Ask the same judge LLM for a final qualitative verdict backed by aggregated scores.

In [ ]:
@traceable(
    name='llm_judge_verdict',
    run_type='chain',
    metadata={'framework': 'Judge', 'node': 'verdict'}
)
def generate_verdict(summary_df) -> str:
    scores_text = summary_df.to_string()
    prompt = (
        'You are an expert AI systems evaluator comparing LangGraph and Google ADK '
        'for educational AI at Mae Fah Luang University.\n\n'
        'Based on the following aggregated LLM-as-a-Judge scores (5 dimensions, 1-5):\n\n'
        f'{scores_text}\n\n'
        'Write a concise final verdict (200-300 words) covering:\n'
        '1. Which framework performed better overall and why\n'
        '2. Specific strengths and weaknesses of each framework\n'
        '3. Recommendation for educational AI use cases like MFU Teacher Assistant\n'
        '4. Any dimension where they were equivalent\n'
        'Be specific and data-driven. Reference the actual scores.'
    )
    return judge_llm.invoke([HumanMessage(content=prompt)]).content

verdict = generate_verdict(overall)
print('=' * 65)
print('  FINAL LLM-AS-A-JUDGE VERDICT')
print('=' * 65)
print(verdict)

## 9. Export Full Report

In [ ]:
with open('judge_report.md', 'w') as f:
    f.write('# LLM-as-a-Judge Evaluation Report\n')
    f.write('## LangGraph vs Google ADK -- AI Teacher Assistant\n\n')
    f.write(f'Generated: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")} UTC\n\n---\n\n')
    f.write('## Overall Framework Scores\n\n')
    f.write(overall.to_markdown() + '\n\n---\n\n')
    f.write('## Per-Scenario Breakdown\n\n')
    f.write(summary.to_markdown() + '\n\n---\n\n')
    f.write('## Final Verdict\n\n')
    f.write(verdict + '\n\n---\n\n')
    f.write('## Individual Run Reasoning\n\n')
    for _, row in df.iterrows():
        f.write(f'### {row["framework"]} -- {row["scenario"]}\n')
        f.write(f'- **Composite:** {row["composite"]}\n')
        f.write(f'- **Reasoning:** {row["reasoning"]}\n')
        f.write(f'- **Strengths:** {row["strengths"]}\n')
        f.write(f'- **Weaknesses:** {row["weaknesses"]}\n\n')

print('Saved: judge_report.md')
files.download('judge_report.md')